In [28]:
import pandas as pd

In [29]:
df = pd.read_csv("../data/raw/flight_data_2024.csv")

print(f"Shape: {df.shape}")
df.head()

C:\Users\mtawa\AppData\Local\Temp\ipykernel_22056\2366555837.py:1: DtypeWarning: Columns (0: cancellation_code) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/raw/flight_data_2024.csv")


Shape: (7079081, 35)


,year,month,day_of_month,day_of_week,fl_date,op_unique_carrier,op_carrier_fl_num,origin,origin_city_name,origin_state_nm,...,diverted,crs_elapsed_time,actual_elapsed_time,air_time,distance,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay
0,2024,1,1,1,2024-01-01,9E,4814.0,JFK,"New York, NY",New York,...,0,136.0,122.0,84.0,509.0,0,0,0,0,0
1,2024,1,1,1,2024-01-01,9E,4815.0,MSP,"Minneapolis, MN",Minnesota,...,0,130.0,114.0,88.0,622.0,0,0,0,0,0
2,2024,1,1,1,2024-01-01,9E,4817.0,JFK,"New York, NY",New York,...,0,106.0,90.0,61.0,288.0,0,0,0,0,0
3,2024,1,1,1,2024-01-01,9E,4817.0,RIC,"Richmond, VA",Virginia,...,0,111.0,76.0,51.0,288.0,0,0,0,0,0
4,2024,1,1,1,2024-01-01,9E,4818.0,DTW,"Detroit, MI",Michigan,...,0,79.0,70.0,45.0,237.0,0,0,0,0,0


In [30]:
# Missing values
print("=== Missing Values ===")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_report[missing_report['Missing Count'] > 0].sort_values('Missing %', ascending=False))

print(df['origin_state_nm'].unique())
# Data types
print("\n=== Data Types ===")
print(df.dtypes)

=== Missing Values ===
                     Missing Count  Missing %
cancellation_code          6982766      98.64
air_time                    113814       1.61
actual_elapsed_time         113814       1.61
arr_delay                   113814       1.61
arr_time                     97854       1.38
wheels_on                    97856       1.38
taxi_in                      97856       1.38
wheels_off                   95734       1.35
taxi_out                     95734       1.35
dep_delay                    92970       1.31
dep_time                     92659       1.31
op_carrier_fl_num                1       0.00
crs_elapsed_time                 1       0.00
<ArrowStringArray>
[                                      'New York',
                                      'Minnesota',
                                       'Virginia',
                                       'Michigan',
                                        'Florida',
                                 'South Carolina',
        

In [31]:
# Step 1 — Drop useless columns
cols_to_drop = [
    'cancellation_code',   # 98.64% missing
    'year',                # only one value
    'day_of_month',        # not in feature list
    'origin_city_name',   # redundant with origin IATA
    'dest_city_name', 'dest_state_nm',       # redundant with dest IATA
    'op_carrier_fl_num',   # flight number, not predictive
    # Operational columns (not known at booking time = leakage)
    'dep_time', 'wheels_off', 'wheels_on',
    'taxi_out', 'taxi_in', 'arr_time',
    'actual_elapsed_time', 'air_time',
    # Delay breakdown (known only AFTER landing = leakage)
    'carrier_delay', 'weather_delay', 'nas_delay',
    'security_delay', 'late_aircraft_delay',
    'dep_delay'            # known at departure not at booking
]

df.drop(columns=cols_to_drop, inplace=True)
print(f"Shape after dropping columns: {df.shape}")
print(f"Remaining columns: {list(df.columns)}")

Shape after dropping columns: (7079081, 14)
Remaining columns: ['month', 'day_of_week', 'fl_date', 'op_unique_carrier', 'origin', 'origin_state_nm', 'dest', 'crs_dep_time', 'crs_arr_time', 'arr_delay', 'cancelled', 'diverted', 'crs_elapsed_time', 'distance']


In [32]:
# Step 2 — Drop cancelled and diverted flights (no arrival delay = useless for prediction)
df = df[(df['cancelled'] == 0) & (df['diverted'] == 0)]
df.drop(columns=['cancelled', 'diverted'], inplace=True)

print(f"Shape after dropping cancelled/diverted: {df.shape}")
print(f"Remaining nulls:\n{df.isnull().sum()}")

Shape after dropping cancelled/diverted: (6965267, 12)
Remaining nulls:
month                0
day_of_week          0
fl_date              0
op_unique_carrier    0
origin               0
origin_state_nm      0
dest                 0
crs_dep_time         0
crs_arr_time         0
arr_delay            0
crs_elapsed_time     0
distance             0
dtype: int64


In [33]:
# Step 3 — Engineer features
df['dep_hour'] = (df['crs_dep_time'] // 100).astype(int)  # e.g. 1415 → 14
df['is_weekend'] = df['day_of_week'].isin([6, 7]).astype(int)  # 6=Sat, 7=Sun

# Step 4 — Create target variable
df['is_delayed'] = (df['arr_delay'] > 15).astype(int)

# Step 5 — Drop columns used only for engineering
df.drop(columns=['crs_dep_time', 'crs_arr_time', 'arr_delay'], inplace=True)

print(f"\nFinal shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nTarget distribution:\n{df['is_delayed'].value_counts(normalize=True).round(4) * 100}")
df.head()


Final shape: (6965267, 12)

Columns: ['month', 'day_of_week', 'fl_date', 'op_unique_carrier', 'origin', 'origin_state_nm', 'dest', 'crs_elapsed_time', 'distance', 'dep_hour', 'is_weekend', 'is_delayed']

Target distribution:
is_delayed
0    79.85
1    20.15
Name: proportion, dtype: float64


,month,day_of_week,fl_date,op_unique_carrier,origin,origin_state_nm,dest,crs_elapsed_time,distance,dep_hour,is_weekend,is_delayed
0,1,1,2024-01-01,9E,JFK,New York,DTW,136.0,509.0,12,0,0
1,1,1,2024-01-01,9E,MSP,Minnesota,CLE,130.0,622.0,10,0,0
2,1,1,2024-01-01,9E,JFK,New York,RIC,106.0,288.0,14,0,0
3,1,1,2024-01-01,9E,RIC,Virginia,JFK,111.0,288.0,16,0,0
4,1,1,2024-01-01,9E,DTW,Michigan,MKE,79.0,237.0,10,0,0


In [34]:
# Rename to match report
df.rename(columns={'op_unique_carrier': 'airline'}, inplace=True)

cols_order = [
    'airline',       # 1 - Categorical
    'origin',        # 2 - Categorical
    'dest',          # 3 - Categorical
    'distance',      # 4 - Numerical
    'dep_hour',      # 5 - Numerical
    'day_of_week',   # 6 - Categorical
    'month',         # 7 - Categorical
    'is_weekend',    # 8 - Binary
    'fl_date',
    'origin_state_nm',
    'is_delayed'     # Target
]

df = df[cols_order]

print(f"Final columns: {list(df.columns)}")
print(f"Shape: {df.shape}")
df.head()

Final columns: ['airline', 'origin', 'dest', 'distance', 'dep_hour', 'day_of_week', 'month', 'is_weekend', 'fl_date', 'origin_state_nm', 'is_delayed']
Shape: (6965267, 11)


,airline,origin,dest,distance,dep_hour,day_of_week,month,is_weekend,fl_date,origin_state_nm,is_delayed
0,9E,JFK,DTW,509.0,12,1,1,0,2024-01-01,New York,0
1,9E,MSP,CLE,622.0,10,1,1,0,2024-01-01,Minnesota,0
2,9E,JFK,RIC,288.0,14,1,1,0,2024-01-01,New York,0
3,9E,RIC,JFK,288.0,16,1,1,0,2024-01-01,Virginia,0
4,9E,DTW,MKE,237.0,10,1,1,0,2024-01-01,Michigan,0


In [35]:
# Save clean dataset
output_path = "../data/processed/flight_features.csv"
df.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
print(f"Shape: {df.shape}")
print(f"\nFinal columns: {list(df.columns)}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nTarget distribution:\n{df['is_delayed'].value_counts()}")

Saved: ../data/processed/flight_features.csv
Shape: (6965267, 11)

Final columns: ['airline', 'origin', 'dest', 'distance', 'dep_hour', 'day_of_week', 'month', 'is_weekend', 'fl_date', 'origin_state_nm', 'is_delayed']

Missing values:
airline            0
origin             0
dest               0
distance           0
dep_hour           0
day_of_week        0
month              0
is_weekend         0
fl_date            0
origin_state_nm    0
is_delayed         0
dtype: int64

Target distribution:
is_delayed
0    5561879
1    1403388
Name: count, dtype: int64
